In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# §6.20 Identical Particles, Exchange Symmetry, and the Pauli Principle

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.20",
    title="Identical Particles, Exchange Symmetry, and the Pauli Principle",
    blurb="Why matter has structure. Two electrons are not just alike — they are "
    "identical, indistinguishable in a way that forces their joint state to be either "
    "symmetric or antisymmetric under exchange. From the antisymmetric case comes the "
    "Pauli principle and, with it, the periodic table, the solidity of solids, and the "
    "pressure that holds up a dying star. And from exchange alone, with no force to "
    "cause it, comes an energy that depends on spin — the root of magnetism and the "
    "chemical bond.",
    difficulty="advanced",
    estimate="170–210 min",
)

## Notebook overview

This is the notebook that explains why matter has the structure it does — the deepest "why is the world
this way" question the volume answers — and it grows from a single, almost trivial-sounding fact: two
electrons are not merely *similar*, they are **identical**. There is no measurement, even in principle,
that can tell one electron from another; they carry no labels. Classically we could still track them
("particle 1 started here, particle 2 there"), but quantum-mechanically we cannot, and quantum mechanics
enforces this indistinguishability with a postulate that turns out to build the periodic table.

The last notebook glimpsed the seed: two spin-$\tfrac12$'s combined into a *symmetric* triplet and an
*antisymmetric* singlet. Here that exchange symmetry is elevated from a feature of coupled states into a
**law**. Because swapping two identical particles can change nothing observable, the state must be an
eigenstate of the exchange operator, and there are only two possibilities: totally **symmetric** or
totally **antisymmetric**. Nature uses both — particles of integer spin are **bosons** (symmetric),
particles of half-integer spin are **fermions** (antisymmetric), the *spin–statistics connection* — and
electrons, being spin-$\tfrac12$, are fermions.

Three consequences follow, and we compute each. First, the **exchange hole**: the antisymmetric state
*vanishes* when two fermions coincide (and when they would occupy the same state), while the symmetric
state is *enhanced* there — a purely quantum correlation with no force behind it. Second, and genuinely
new, the **exchange interaction**: even with a Hamiltonian carrying *no* spin-dependent force, symmetric
and antisymmetric spatial states have different average separations and hence different interaction
energies — and because an electron pair's overall state must be antisymmetric, the *spatial* symmetry is
locked to the *spin* state (symmetric-spatial ↔ singlet, antisymmetric-spatial ↔ triplet, [§6.19](addition-angular-momenta.ipynb)). So the
energy depends on spin though the Hamiltonian does not: this is the origin of **Hund's rule**,
**ferromagnetism**, and the **covalent bond** — magnetism that is electrostatic in origin, filtered
through exchange symmetry. Third, the **Slater determinant** packages antisymmetry into one object that
vanishes automatically if two orbitals coincide — the **Pauli exclusion principle** — which forces
electrons into successive shells, giving the periodic table, the size and rigidity of matter, and the
degeneracy pressure that holds up white dwarfs.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the two-particle states via `numpy.outer` of single-particle
box orbitals ([§6.10](schrodinger-on-a-computer.ipynb)), `numpy.linalg.det` for the Slater determinant, and 2-D `numpy.trapezoid` sums for
$\langle(x_1-x_2)^2\rangle$ and $\langle V_{\text{int}}\rangle$.

> **Scope and conventions.** Single-particle states are the box eigenstates of [§6.10](schrodinger-on-a-computer.ipynb) ($\varphi_n(x)=
> \sqrt{2/L}\sin(n\pi x/L)$ on $[0,L]$, $L=1$). Two-particle wavefunctions are spatial only; the spin
> state is implied by the requirement that the *overall* fermion state be antisymmetric. **This notebook
> establishes the microscopic rule** (symmetrization, the exchange interaction, Slater/Pauli) **and its
> consequences for the structure of matter** (the periodic table, bonding, magnetism, stellar stability).
> It does **not** compute the quantum *statistics* — the Fermi–Dirac and Bose–Einstein *distributions*,
> quantum gases, blackbody radiation, Bose–Einstein condensation — which are the thermodynamics **built
> on** this rule, deferred to **Volume VII**. Indistinguishability is *why* quantum particles are counted
> differently from classical ones (the $1/N!$ and the correlations that Volume V's counting, [§5.1](../05-classical-stat-mech/counting.ipynb), could
> only anticipate); here is the microscopic foundation, and Volume VII builds the statistics on it. See
> Sakurai & Napolitano and Griffiths (identical particles, exchange, Slater determinants); and Notebooks
> [§6.19](addition-angular-momenta.ipynb) (singlet/triplet), [§6.8](bloch-sphere-entanglement.ipynb) (entanglement), [§6.17](hydrogen-atom.ipynb)/[§6.18](spin-magnetic.ipynb) (the $2n^2$ states), [§6.10](schrodinger-on-a-computer.ipynb) (the box orbitals),
> [§5.1](../05-classical-stat-mech/counting.ipynb) (counting).

## Theory in brief

### Indistinguishability and the two allowed symmetries

Exchanging two identical particles must leave every observable unchanged, so the exchange operator
$P_{12}$ (swap the particles) commutes with $H$ and every observable; since $P_{12}^2=I$, its eigenvalues
are $\pm1$, and physical states must be eigenstates:

```{math}
:label: eq-indistinguishable
P_{12}\Psi=\pm\Psi:\qquad \text{symmetric } (+1)\ \text{ or }\ \text{antisymmetric } (-1) .
```

### The symmetrization postulate and spin–statistics

Which sign a given species uses is not decided by {eq}`eq-indistinguishable`: nothing in
non-relativistic quantum mechanics ties the exchange eigenvalue to any other property of the particle.
Nature nevertheless makes a universal choice, taken here as a postulate:

```{math}
:label: eq-symmetrization
\text{integer spin}\Rightarrow\text{BOSONS (symmetric)},\qquad \text{half-integer spin}\Rightarrow\text{FERMIONS (antisymmetric)} ,
```

the **spin–statistics connection** — a theorem of relativistic quantum field theory (named here as a
horizon, not proved). Electrons, protons, neutrons are fermions; photons and many nuclei are bosons.

### Two-particle states and the exchange hole

From single-particle states $\varphi_a,\varphi_b$,

```{math}
:label: eq-two-particle
\Psi_{S/A}(x_1,x_2)=\frac{\varphi_a(x_1)\varphi_b(x_2)\pm\varphi_b(x_1)\varphi_a(x_2)}{\sqrt2} ,
```

with $+$ symmetric (bosons) and $-$ antisymmetric (fermions). $\Psi_A$ **vanishes** when $x_1=x_2$ (and
when $a=b$): fermions cannot coincide or share a state — the **exchange hole**. $\Psi_S$ is **enhanced**
there: bosons bunch. This is a correlation with *no force* behind it.

### The exchange interaction

Even with a spin-independent $H$, the average separation differs (Griffiths computes the three
expectation values in full):

```{math}
:label: eq-exchange
\langle(x_1-x_2)^2\rangle_S<\langle(x_1-x_2)^2\rangle_{\text{dist}}<\langle(x_1-x_2)^2\rangle_A ,
```

so with a repulsion $V(x_1,x_2)$ the antisymmetric state (particles farther) has lower energy. For
electrons the overall state is antisymmetric, tying spatial symmetry to spin (symmetric-spatial ↔
singlet, antisymmetric-spatial ↔ triplet, [§6.19](addition-angular-momenta.ipynb)), so the energy depends on spin though $H$ does not — the
**exchange interaction**, the origin of Hund's rule, ferromagnetism, and the covalent bond. *Magnetism is
electrostatic repulsion filtered through exchange symmetry.*

### The Slater determinant and Pauli exclusion

Antisymmetrizing $N$ particles by hand means summing the $N!$ permutations of the product
$\varphi_1\cdots\varphi_N$ with alternating signs: precisely the expansion of a determinant. The
$N$-fermion generalization of {eq}`eq-two-particle` is therefore

```{math}
:label: eq-slater
\Psi(x_1,\dots,x_N)=\frac{1}{\sqrt{N!}}\det\big[\varphi_i(x_j)\big] ,
```

automatically antisymmetric (swap two particles → swap two columns → sign flip), and **zero if two
orbitals coincide** (two equal rows) — the **Pauli exclusion principle**: no two identical fermions in
the same single-particle state.

### The architecture of matter

We now aim the exclusion principle at the atom. The one-electron states of [§6.17](hydrogen-atom.ipynb) carry the labels
$(n,l,m_l)$, spin ([§6.18](spin-magnetic.ipynb)) doubles them with $m_s$, and Pauli admits at most one electron per label, so

```{math}
:label: eq-matter
\text{fill } (n,l,m_l,m_s)\ \text{one electron each}\ \Rightarrow\ 2n^2\ \text{per shell}=2,8,18,32 ,
```

the periodic table. Exclusion also gives matter its **size and rigidity** (a degeneracy pressure —
electrons cannot all fall to the ground state) and holds up **white dwarfs** and **neutron stars**. And
indistinguishability is *why* quantum particles are counted differently from classical ones ([§5.1](../05-classical-stat-mech/counting.ipynb)) — the
microscopic foundation on which Volume VII's quantum statistics is built.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED, BLUE = "#c1121f", "#1d4e89"

# the single-particle box: [0, L], with the §6.10 box eigenstates φ_n(x) = √(2/L) sin(nπx/L)
L = 1.0
N_GRID = 400
x = np.linspace(0, L, N_GRID)
X1, X2 = np.meshgrid(x, x, indexing="ij")


def box_orbital(n):
    """The n-th particle-in-a-box eigenstate φ_n(x)=√(2/L) sin(nπx/L) (the §6.10 box states), normalized."""
    f = np.sqrt(2 / L) * np.sin(n * np.pi * x / L)
    return f / np.sqrt(np.trapezoid(f**2, x))


def two_particle_state(phi_a, phi_b, symmetry):
    r"""Build a normalized two-particle state on the $(x_1,x_2)$ grid via ``numpy.outer`` {eq}`eq-two-particle`.

    ``symmetry='S'`` gives the symmetric $\Psi_S$ (bosons), ``'A'`` the antisymmetric $\Psi_A$ (fermions),
    ``'D'`` the distinguishable product $\varphi_a(x_1)\varphi_b(x_2)$ (the classical reference).
    """
    prod_ab = np.outer(phi_a, phi_b)  # φ_a(x1) φ_b(x2)
    prod_ba = np.outer(phi_b, phi_a)  # φ_b(x1) φ_a(x2)
    if symmetry == "S":
        psi = (prod_ab + prod_ba) / np.sqrt(2)
    elif symmetry == "A":
        psi = (prod_ab - prod_ba) / np.sqrt(2)
    else:  # 'D' — distinguishable product state
        psi = prod_ab
    norm = np.sqrt(np.trapezoid(np.trapezoid(np.abs(psi) ** 2, x, axis=1), x))
    return psi / norm


def exchange_expectation(Psi, operator2d):
    r"""The expectation $\langle O\rangle=\iint|\Psi|^2 O(x_1,x_2)\,dx_1dx_2$ by 2-D ``numpy.trapezoid``."""
    integrand = np.abs(Psi) ** 2 * operator2d
    return np.trapezoid(np.trapezoid(integrand, x, axis=1), x)


def slater_determinant(orbitals, points):
    r"""The Slater determinant $\det[\varphi_i(x_j)]$ of single-particle ``orbitals`` at ``points`` {eq}`eq-slater`.

    ``orbitals`` is a list of callables $\varphi_i$ (functions of a scalar position), ``points`` the
    particle positions $x_j$; returns $\det$ of the matrix $M_{ij}=\varphi_i(x_j)$ (``numpy.linalg.det``).
    Antisymmetric by construction, and zero if two orbitals (rows) coincide — the Pauli principle.
    """
    M = np.array([[orb(p) for p in points] for orb in orbitals])
    return np.linalg.det(M)

## Exercise 1 — Exchange symmetry and the two allowed states

Build the symmetric and antisymmetric two-particle states from two box orbitals, and confirm their
exchange symmetry $\Psi_S(x_2,x_1)=+\Psi_S$, $\Psi_A(x_2,x_1)=-\Psi_A$ {eq}`eq-indistinguishable`,
{eq}`eq-symmetrization`.

1. Take two single-particle box states $\varphi_a=\varphi_1$, $\varphi_b=\varphi_2$.
2. Form $\Psi_S$ and $\Psi_A$ with `numpy.outer` (the `two_particle_state` helper).
3. Verify the symmetry under $x_1\leftrightarrow x_2$ (transposing the 2-D array).
4. Note bosons use $\Psi_S$, fermions $\Psi_A$ (spin–statistics). Frame: only two exchange
   symmetries exist.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    sym_ok and antisym_ok,
    "identical-particle states are symmetric (bosons) or antisymmetric (fermions) under exchange",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The exchange hole

Show quantitatively that fermions avoid each other and bosons cluster, before any interaction:
$\Psi_A$ vanishes on the coincidence diagonal $x_1=x_2$, while $\Psi_S$ is enhanced there
{eq}`eq-two-particle`.

1. Evaluate $|\Psi_A|^2$ and $|\Psi_S|^2$ on the diagonal $x_1=x_2$ (the diagonal of the 2-D
   array).
2. Confirm $\Psi_A=0$ there (the exchange hole) and $\Psi_S>0$ (bunching).
3. Interpret as a statistical correlation from symmetry alone. Frame: the Pauli hole and bosonic
   bunching, before any force.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    diag_A.max(),
    0.0,
    "the antisymmetric state vanishes when the particles coincide — the exchange hole (Pauli, before any force)",
    atol=1e-6,
)

## Exercise 3 — The exchange effect on separation

Compute the average squared separation $\langle(x_1-x_2)^2\rangle$ for the symmetric,
antisymmetric, and distinguishable states, and show symmetric $<$ distinguishable $<$
antisymmetric — symmetry acts like a force {eq}`eq-exchange`.

1. Build the operator $(x_1-x_2)^2$ on the grid.
2. Compute $\langle(x_1-x_2)^2\rangle$ for $\Psi_S$, $\Psi_A$, and the distinguishable product
   $\Psi_D$ (the `exchange_expectation` helper, 2-D `numpy.trapezoid`).
3. Confirm the ordering.
4. Interpret as an effective attraction (symmetric) or repulsion (antisymmetric) with *no force*
   in $H$. Frame: symmetry acts like a force.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    sep_ok,
    "⟨(x₁−x₂)²⟩ is smallest for symmetric and largest for antisymmetric — exchange pulls symmetric states together and pushes antisymmetric apart",
)

## Exercise 4 — The exchange energy

With a repulsive interaction, compute the energy difference between the symmetric and
antisymmetric states — the **exchange energy** — and connect it to spin through the overall
antisymmetry {eq}`eq-exchange`.

1. Add a (soft) repulsion $V(x_1,x_2)=1/\sqrt{(x_1-x_2)^2+\epsilon^2}$.
2. Compute $\langle V\rangle$ for $\Psi_S$ and $\Psi_A$ (the `exchange_expectation` helper).
3. Show the antisymmetric state (particles farther) has *lower* interaction energy.
4. Connect to electrons: the overall antisymmetry ties spatial symmetry to spin (symmetric-spatial
   ↔ singlet, antisymmetric-spatial ↔ triplet, [§6.19](addition-angular-momenta.ipynb)), so the energy depends on spin though $V$
   does not. Frame: Hund's rule, magnetism, the covalent bond — spin-dependent energy from a
   spin-independent Hamiltonian.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    exchange_ok,
    "⟨V_int⟩ differs between Ψ_S and Ψ_A (the exchange energy) — a spin-dependent energy with no spin-dependent force",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The Slater determinant and Pauli exclusion

Build the antisymmetric $N$-fermion state as a **Slater determinant** and show it vanishes when
two orbitals coincide — the **Pauli exclusion principle** {eq}`eq-slater`.

1. Form the matrix $M_{ij}=\varphi_i(x_j)$ for distinct orbitals and take its determinant
   (`numpy.linalg.det`, the `slater_determinant` helper).
2. Verify swapping two particles (two columns) flips the sign.
3. Show that with two *identical* orbitals (two equal rows) the determinant is zero.
4. State the Pauli exclusion principle. Frame: antisymmetry and exclusion in one object.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    det_repeated,
    0.0,
    "the Slater determinant vanishes when two orbitals coincide — the Pauli exclusion principle",
    atol=1e-9,
)

## Exercise 6 — Building the periodic table *(student)*

Fill the hydrogenic states one electron at a time by exclusion and reproduce the shell structure,
recovering the $2n^2=2,8,18,32$ capacities {eq}`eq-matter`.

1. List the states $(n,l,m_l,m_s)$ of each shell $n$.
2. Place one electron per state (Pauli).
3. Count the electrons at each shell closing — the $2n^2$ capacities.
4. Note the outer electrons set chemistry, and that exclusion is *why* electrons do not all
   collapse to $n=1$ (matter has size). Frame: the architecture of the periodic table from one
   rule.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    capacities_ok,
    "filling states one-per-(n,l,m_l,m_s) gives the 2n²=2,8,18,32 shell structure — the Pauli principle builds the periodic table",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Bosons versus fermions in a trap *(student)*

Contrast how two bosons and two fermions occupy the low-lying states of the box, and compare their
ground-state energies {eq}`eq-symmetrization`, {eq}`eq-two-particle`.

1. For two **bosons**, both can occupy the ground orbital $\varphi_1$ (symmetric).
2. For two **fermions**, they must occupy *different* orbitals (antisymmetric) — the second goes
   to $\varphi_2$.
3. Compute and compare the total energies ($E_n=n^2$ in box units).
4. Note the fermionic "cost" (the Pauli pressure) and the bosonic tendency to pile into one state
   (anticipating BEC, Volume VII). Frame: the same trap, two utterly different ground states.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    trap_ok,
    "two bosons share the ground orbital while two fermions must fill two orbitals (Pauli) — exchange symmetry dictates how identical particles occupy states",
)

## Exercise 8 — One rule, and the shape of the world

The whole of this notebook grew from a single impossibility — that two identical particles cannot be told
apart — which forced their joint state to be symmetric or antisymmetric and split the particles of the
world into bosons and fermions. From antisymmetry came the **Pauli principle**, and from it the periodic
table, the solidity and size of matter, and the degeneracy pressure inside a white dwarf. From
**exchange** came an energy that depends on spin with no magnetic force behind it, and with it magnetism
and the chemical bond.

**This exercise (synthesis).** No new computation: the structure of matter is the result. **Movement IV
is complete** — we have the electron's spin ([§6.18](spin-magnetic.ipynb)), the coupling of angular momenta ([§6.19](addition-angular-momenta.ipynb)), and now the
law of identical particles (§6.20): everything needed to describe real atoms. We should also be clear
about the boundary we have drawn. This notebook gave the *microscopic rule* — indistinguishability,
symmetrization, the exchange interaction, Pauli exclusion — and its consequences for the *structure* of
matter. Building the *thermodynamics* on it — the Fermi–Dirac and Bose–Einstein distributions, quantum
gases, blackbody radiation, Bose–Einstein condensation — is the work of **Volume VII**, which stands on
exactly this foundation (and completes the indistinguishable-counting that Volume V's [§5.1](../05-classical-stat-mech/counting.ipynb) could only
anticipate). The final movement of *this* volume turns instead to a different practical fact: almost no
real system is exactly solvable, so we need **approximation methods** — perturbation theory, the
variational method, the semiclassical limit — beginning with the fine structure that spin–orbit coupling
adds to the hydrogen levels ([§6.21](perturbation-fine-structure.ipynb)).

It is hard to overstate how much rides on a minus sign. Symmetric or antisymmetric — that single choice
under exchange decides whether particles pile into one state or refuse to share it, and the refusal is
why you are not falling through your chair. The stability of matter is, in the end, a theorem about the
sign of a wavefunction.

## Notebook summary

Identical particles and the Pauli principle — the close of Movement IV, and why matter has structure.

- **Indistinguishability** {eq}`eq-indistinguishable`, {eq}`eq-symmetrization`: states must be symmetric
  (**bosons**, integer spin) or antisymmetric (**fermions**, half-integer spin) — the spin–statistics
  connection (named, not proved).
- **The exchange hole** {eq}`eq-two-particle`: $\Psi_A$ vanishes when fermions coincide, $\Psi_S$ is
  enhanced when bosons do — correlations from symmetry, with no force (`numpy.outer`).
- **The exchange interaction** {eq}`eq-exchange`: $\langle(x_1-x_2)^2\rangle_S<\langle\cdot\rangle_A$, so
  a spin-independent repulsion gives a spin-dependent energy (spatial ↔ spin symmetry, [§6.19](addition-angular-momenta.ipynb)) — Hund's
  rule, magnetism, the covalent bond.
- **The Slater determinant** {eq}`eq-slater`: $\det[\varphi_i(x_j)]$ (`numpy.linalg.det`) is
  antisymmetric and vanishes for a repeated orbital — the **Pauli exclusion principle**.
- **The architecture of matter** {eq}`eq-matter`: filling $(n,l,m_l,m_s)$ by exclusion gives $2n^2=2,8,
  18,32$ — the periodic table, the size of atoms, the pressure in white dwarfs.

One minus sign under exchange builds the periodic table and holds up your chair. Movement IV is complete;
the quantum *statistics* built on this rule is Volume VII.

## Outlook

- **Fine structure ([§6.21](perturbation-fine-structure.ipynb))** — the opening of Movement V: spin–orbit and relativistic corrections to the
  hydrogen levels, by perturbation theory.
- **Quantum statistics (Volume VII)**: the Fermi–Dirac and Bose–Einstein distributions, quantum gases,
  blackbody radiation, Bose–Einstein condensation — the thermodynamics built on this notebook's rule.
- **The many-electron atom and quantum chemistry**: Hartree–Fock, the aufbau/Madelung filling, screening
  (horizons).
- **Degeneracy pressure and the stability of stars**: white dwarfs and neutron stars (a horizon).
- **Cross-reference** [§6.19](addition-angular-momenta.ipynb) (singlet/triplet), [§6.8](bloch-sphere-entanglement.ipynb) (entanglement), [§6.17](hydrogen-atom.ipynb)/[§6.18](spin-magnetic.ipynb) (the $2n^2$ states), [§5.1](../05-classical-stat-mech/counting.ipynb)
  (counting — the classical anticipation), and forward to [§6.21](perturbation-fine-structure.ipynb), Volume VII.

In [ ]:
from ecp.style import footer

footer()